# snowflakeR Quickstart -- Workspace Notebook

This notebook is for **Snowflake Workspace Notebooks** (Python kernel + `%%R` magic).
For local environments (RStudio, Posit, JupyterLab), use `local_quickstart.ipynb`.

**Before you start:** Optionally edit `snowflaker_config.yaml` to set your
database, schema, warehouse, and EAI settings.

**Sections:**
1. Setup (install R + snowflakeR)
2. Connect & set execution context
3. Queries & Table Operations
4. Additional Query Patterns
5. Visualization with ggplot2
6. Cleanup

## 1. Setup

One Python cell to set session context, validate EAI, install R, and
install snowflakeR -- after that, everything is pure R.

Optionally edit `snowflaker_config.yaml` to set `context`, `eai`, or
`tarballs`. If omitted, session defaults are used and EAIs are auto-discovered.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_config.yaml", packages=["snowflakeR"])

## 2. Connect

From here on, it's all R. Session context (database, schema, warehouse)
was set by `setup_notebook()` in the setup cell above.

In [ ]:
%%R
library(snowflakeR)

# Connect (auto-detects Workspace session; context set by setup_notebook)
conn <- sfr_connect()
conn

---
## 3. Queries & Table Operations

In [ ]:
%%R
# Run a SQL query
result <- sfr_query(conn, "SELECT CURRENT_TIMESTAMP() AS now, CURRENT_USER() AS user_name")
result

In [ ]:
%%R
# Write a data.frame to Snowflake (fully qualified name)
sfr_write_table(conn, sfr_fqn(conn, "SFR_MTCARS"), mtcars, overwrite = TRUE)

In [ ]:
%%R
# List tables
tables <- sfr_list_tables(conn)
rcat("Tables:", paste(head(tables, 10), collapse = ",\n  "))

In [ ]:
%%R
# Read it back (fully qualified name)
df <- sfr_read_table(conn, sfr_fqn(conn, "SFR_MTCARS"))
head(df, 5)

In [ ]:
%%R
# Describe columns
sfr_list_fields(conn, sfr_fqn(conn, "SFR_MTCARS"))

---
## 4. Additional Query Patterns

`sfr_query()` returns data.frames. For standard DBI-compliant access and
`dbplyr`/`dplyr` integration, install the companion
[RSnowflake](https://github.com/Snowflake-Labs/RSnowflake) package. See
`RSnowflake/inst/notebooks/demo_rsnowflake.ipynb` for a full walkthrough
(DBI, dbplyr, Arrow, ADBC). Locally, `local_quickstart.ipynb` Section 4 shows
how to bridge via `sfr_dbi_connection()`.

In [ ]:
%%R
# sfr_query returns results as data.frames
result <- sfr_query(conn, "SELECT 42 AS answer")
cat("Table exists:", sfr_table_exists(conn, sfr_fqn(conn, "SFR_MTCARS")), "\n")
result

In [ ]:
%%R
# Aggregate query via sfr_query
summary <- sfr_query(conn, paste(
  "SELECT CYL, COUNT(*) AS n, AVG(MPG) AS avg_mpg, AVG(HP) AS avg_hp",
  "FROM", sfr_fqn(conn, "SFR_MTCARS"),
  "GROUP BY CYL ORDER BY CYL"
))

summary

# For dplyr/dbplyr integration, install RSnowflake and use:
#   dbi_con <- sfr_dbi_connection(conn)
#   tbl(dbi_con, I(sfr_fqn(conn, "SFR_MTCARS"))) |>
#     group_by(CYL) |> summarise(n = n(), avg_mpg = mean(MPG)) |> collect()

---
## 5. Visualization with ggplot2

Use `%%R -w WIDTH -h HEIGHT` and `print(p)` for plots in Workspace Notebooks.

In [ ]:
%%R -w 700 -h 450
library(ggplot2)

df <- sfr_read_table(conn, sfr_fqn(conn, "SFR_MTCARS"))

p <- ggplot(df, aes(x = WT, y = MPG, color = factor(CYL))) +
  geom_point(size = 3) +
  labs(title = "MPG vs Weight by Cylinder Count",
       x = "Weight (1000 lbs)", y = "Miles per Gallon",
       color = "Cylinders") +
  theme_minimal()

print(p)  # print() required in Workspace Notebooks

---
## 6. Cleanup

In [ ]:
%%R
# Uncomment to clean up demo objects
# (commented out to avoid accidental deletion on Run All)
#
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_MTCARS")))
# rcat("Cleanup complete.")

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.QUICKSTART_RESULTS AS
        SELECT 'quickstart' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.QUICKSTART_RESULTS")
except Exception:
    pass